# 01: Capital Performance Overview

**Goal:** Analyze FTA National Transit Database capital expense patterns across agencies, modes, and geographies.

**Data:** FTA NTD Capital Expenses, 2024 — 1,000 agency-mode records, $13.3B total capital
**Source:** https://data.transportation.gov/resource/fphd-jyyj.csv (Socrata)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/ntd_capital_expenses.csv')

# Clean numeric columns
numeric_cols = ['guideway','stations','administrative_buildings','maintenance_buildings',
                'passenger_vehicles','other_vehicles','fare_collection_equipment',
                'communication_information','other','total','primary_uza_population']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f"Records: {len(df)}")
print(f"Agencies: {df['agency'].nunique()}")
print(f"Modes: {df['mode_name'].nunique()}")
print(f"States: {df['state'].nunique()}")
print(f"Total capital: ${df['total'].sum()/1e9:.1f}B")
df.head()

## Capital Spend by Mode

In [ ]:
mode_stats = df.groupby('mode_name').agg({
    'total': ['sum','mean','count'],
    'passenger_vehicles': 'sum',
    'guideway': 'sum'
}).round(0)
mode_stats.columns = ['_'.join(c) for c in mode_stats.columns]
mode_stats = mode_stats.sort_values('total_sum', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mode_stats['total_sum'].plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title('Total Capital by Transit Mode')
axes[0].set_xlabel('USD')

mode_stats['total_mean'].plot(kind='barh', ax=axes[1], color='#3498db')
axes[1].set_title('Mean Capital per Agency-Mode')
axes[1].set_xlabel('USD')

plt.tight_layout()
plt.show()

## Top Agencies by Capital

In [ ]:
agency_total = df.groupby('agency')['total'].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
agency_total.plot(kind='barh', ax=ax, color='#e74c3c')
ax.set_title('Top 15 Agencies by Total Capital Spend')
ax.set_xlabel('USD')
plt.tight_layout()
plt.show()

## Geographic Distribution

In [ ]:
state_stats = df.groupby('state').agg({
    'total': 'sum',
    'agency': 'nunique'
}).sort_values('total', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

state_stats['total'].plot(kind='bar', ax=axes[0], color='#f39c12')
axes[0].set_title('Capital by State')
axes[0].set_xlabel('State')
axes[0].set_ylabel('USD')
axes[0].tick_params(axis='x', rotation=45)

state_stats['agency'].plot(kind='bar', ax=axes[1], color='#9b59b6')
axes[1].set_title('Agency Count by State')
axes[1].set_xlabel('State')
axes[1].set_ylabel('Agencies')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Asset Breakdown

In [ ]:
asset_cols = ['guideway','stations','administrative_buildings','maintenance_buildings',
              'passenger_vehicles','other_vehicles','fare_collection_equipment',
              'communication_information','other']
asset_totals = df[asset_cols].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
asset_totals.plot(kind='barh', ax=ax, color='#1abc9c')
ax.set_title('Capital by Asset Category (All Agencies)')
ax.set_xlabel('USD')
plt.tight_layout()
plt.show()